In [ ]:
import os
import sys
import h5py
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(".."))

from toolbox_mingyang import *

f = h5py.File("../data/SimuData.mat", "r")
print(list(f.keys()))


In [ ]:
Mag = f["Magnitude"][:]

Npe, Nro = Mag.shape

show_mri(Mag, "mag")

In [ ]:
def simulate_coils(Nro, Npe, n_coils = 8):
    """
    Input   : (Npe Nro)
    Output  : (Nco, Mpe, Nro)
    """

    x = np.linspace(-1, 1, Nro)
    y = np.linspace(-1, 1, Npe)
    X, Y = np.meshgrid(x, y)

    csm = []

    for i in range(n_coils):
        angle = 2*np.pi*i/n_coils
        cx, cy = 0.5*np.cos(angle), 0.5*np.sin(angle)

        sens = np.exp(-((X-cx)**2 + (Y-cy)**2)/0.3)
        csm.append(sens)

    return np.stack(csm, axis=0)


In [ ]:
coil_imgs = simulate_coils(Nro, Npe, 8)
for i in range(8):
    show_mri(coil_imgs[i], title=f"Coil {i}")

In [ ]:
Multi_coil_img = coil_imgs * Mag
for i in range(8):
    show_mri(Multi_coil_img[i], title=f"Coil {i}")


In [ ]:
def sos_combine(coil_imgs, axis=0, keepdims=False):
    """
    Sum-of-Squares combine

    coil_imgs: np.ndarray
    axis: 沿哪个维度做 SoS（通常是 coil 维度）
    keepdims: 是否保留该维度

    return: 合并后的图像
    """
    return np.sqrt(np.sum(np.abs(coil_imgs)**2, axis=axis, keepdims=keepdims))

In [ ]:
img = sos_combine(coil_imgs, axis=0)
print(img.shape)
show_mri(img, "mag")